# Lenguajes y Autómatas II (SCD-1016)
## Sesión 13 · Manejo de Ámbitos (Scope Global vs Local)

**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`) · **Repositorio maestro:** https://github.com/ijmarquezl/tecnm-lenguajes-automatas-2

---

> **Pregunta guía:** ¿Cómo sabe el compilador cuál variable "x" estás usando si hay varias con ese nombre?
>
> **Objetivo de la sesión:** Construir una pila de ámbitos enlazados y buscar un identificador desde el bloque actual hacia la raíz, respetando el ocultamiento.

**Entregable:** tus funciones declarar() y buscar() en artesano_sesion13_c.c, la salida del harness y la bitácora en auditorias/auditoria_sesion13.md

**Tiempo:** 60 min · **Fases:** Artesano (manual) → Científico (IA con contrato) → Auditor (harness)


## 1. ¿Qué vas a construir?

Recibes ámbitos enlazados ya inicializados y una lista de nombres; declarar() los guarda solo en el ámbito actual y buscar() devuelve el nombre del ámbito donde se encontró, empezando por el actual. Si el nombre no existe devuelve NULL.

Al final de la sesión tendrás: **Una pila de ámbitos enlazada que declara nombres y los busca desde el bloque actual hasta la raíz, con el harness en verde.**

Los conceptos y el porqué de esta sesión están en la presentación
([sesión 13](https://ijmarquezl.github.io/tecnm-lenguajes-automatas-2/presentaciones/sesion13.html)). Aquí solo está el trabajo: qué se
recibe, qué debe salir y cómo se comprueba.


## 2. El contrato exacto

El archivo `api_sesion13.h` declara la interfaz que el harness sabe probar.
Tu implementación **debe** respetar esas firmas; si las cambias, el harness no compila.

```c
#include "api_sesion13.h"

int declarar(Ambito *ambito, const char *nombre);
const char *buscar(const Ambito *ambito, const char *nombre);
```

- declarar() no consulta los ámbitos padre: el mismo nombre puede existir una vez en cada nivel.
- declarar() devuelve 1 si insertó, 0 si el nombre ya estaba en ese ámbito y -1 si el ámbito está lleno o recibe NULL o un nombre vacío.
- buscar() devuelve el nombre del ámbito más cercano, no el símbolo encontrado.

> El harness compila con `-Wall -Wextra -Werror`: una advertencia del compilador cuenta como error.
>
> El harness compila `harness_sesion13_c.c` junto con `artesano_sesion13_c.c`
> (o `ia_sesion13_c.c`). Por eso tu archivo **no debe tener `main()`**: el `main()`
> es del harness.


## 3. Cómo se califica

### Casos que evalúa el harness

| # | Caso | Resultado esperado |
| :-: | :--- | :--- |
| 1 | `declarar(&g, "x")` | 1 |
| 2 | `buscar en el mismo ambito` | `global` |
| 3 | `buscar subiendo al padre` | `global` |
| 4 | `shadowing en el hijo` | `bloque` |
| 5 | `tres niveles hasta la raiz` | `raiz` |
| 6 | `nombre no declarado` | `(nulo)` |

### Salida literal que debes obtener

```text
=== AUDITORIA SESION 13 - Pila de ambitos y busqueda jerarquica ===
Caso                               esperado:              obtenido:              resultado
[ 1] declarar(&g, "x")                  esperado: 1            obtenido: 1            OK
[ 2] buscar en el mismo ambito          esperado: global       obtenido: global       OK
[ 3] buscar subiendo al padre           esperado: global       obtenido: global       OK
[ 4] shadowing en el hijo               esperado: bloque       obtenido: bloque       OK
[ 5] tres niveles hasta la raiz         esperado: raiz         obtenido: raiz         OK
[ 6] nombre no declarado                esperado: (nulo)       obtenido: (nulo)       OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion13.md
```

Los números entre corchetes son el contador de casos. Mientras algo falle verás
`FALLA` en esa línea, el total será menor y el programa terminará con código de
salida 1.

### Criterios de aceptación

- [ ] Cada nombre queda registrado solo en el ámbito donde se declara.
- [ ] buscar() devuelve el ámbito más interno que declara el nombre y sube hasta la raíz cuando hace falta.
- [ ] Un nombre inexistente devuelve NULL sin modificar los ámbitos.


## Paso 0 · Entorno


In [ ]:
import os

if os.system('which gcc > /dev/null 2>&1') != 0:
    print('⏳ Instalando compilador...')
    !apt-get update -qq && apt-get install -y -qq gcc > /dev/null 2>&1

print('✅ Entorno listo:')
!gcc --version | head -n 1


## Fase 1 · El Artesano (15 min, sin IA)

Escribe a mano la implementación en `artesano_sesion13_c.c`. El archivo ya trae la
firma y un `TODO`: edita el contenido de la celda `%%writefile` que está abajo (o el
archivo en el panel **Archivos** de Colab) y vuelve a ejecutarla para guardar los
cambios. Hazlo sin consultar a la IA. Si te atoras, anota en tu bitácora exactamente
en qué te atoraste; eso también es evidencia.


In [ ]:
%%writefile api_sesion13.h
/* api_sesion13.h - contrato de la Sesion 13
 * Pila de ámbitos y búsqueda jerárquica
 *
 * Esta cabecera es el contrato: el harness y tu implementacion
 * deben incluirla. Si cambias una firma, el harness dejara de compilar.
 */
#ifndef API_SESION13_H
#define API_SESION13_H

#define AMBITO_MAX 8
typedef struct Ambito {
    const char *nombre;
    const char *nombres[AMBITO_MAX];
    int total;
    struct Ambito *padre;
} Ambito;

int declarar(Ambito *ambito, const char *nombre);
const char *buscar(const Ambito *ambito, const char *nombre);

#endif /* API_SESION13_H */


In [ ]:
%%writefile artesano_sesion13_c.c
/* artesano_sesion13_c.c - Fase 1: El Artesano (sin IA)
 * Pila de ámbitos y búsqueda jerárquica.
 *
 * Implementa lo que declara api_sesion13.h.
 * El harness compila este archivo junto con el suyo: no escribas main().
 */
#include "api_sesion13.h"
#include <stdio.h>
#include <string.h>

int declarar(Ambito *ambito, const char *nombre) {
    (void)ambito; (void)nombre; /* TODO: guarda el nombre si no esta en este ambito */
    return -1;
}

const char *buscar(const Ambito *ambito, const char *nombre) {
    (void)ambito; (void)nombre; /* TODO: sube por padre hasta encontrar el nombre */
    return NULL;
}


## Fase 2 · El Científico (10 min)

Redacta el contrato **C-R-E-O** (Contexto, Rol, Especificación, Output) para pedirle
la misma función a la IA, y pega el resultado en la celda de `ia_sesion13_c.c`. La
firma debe ser idéntica a la del contrato; si la IA la cambia, corrígela antes de
compilar.


In [ ]:
%%writefile ia_sesion13_c.c
/* ia_sesion13_c.c - Fase 2: El Cientifico
 *
 * Pega aqui el codigo que te devolvio la IA, con la MISMA firma
 * que declara api_sesion13.h. No puede traer main().
 */
#include "api_sesion13.h"

/* TODO: pega aqui la propuesta de la IA */


## Fase 3 · El Auditor (10 min)

El harness de abajo **no aprueba nada por sí solo**: ejecuta tu función caso por caso,
la compara contra el valor esperado e imprime `OK` o `FALLA`. Léelo antes de ejecutarlo:
los casos te dicen qué se está midiendo.


In [ ]:
%%writefile harness_sesion13_c.c
/* harness_sesion13_c.c - auditoria automatica de la Sesion 13
 *
 * Compilacion:
 *   gcc -std=c11 -Wall -Wextra -Werror harness_sesion13_c.c artesano_sesion13_c.c -o test_artesano
 *
 * El harness NO aprueba nada por si solo: cada linea se calcula
 * ejecutando tu implementacion y comparandola contra el valor esperado.
 * Si algun caso falla, el programa termina con codigo de salida 1.
 */
#include <stdio.h>
#include <string.h>
#include "api_sesion13.h"

static int probadas = 0;
static int fallidas = 0;

static void verificar_str(const char *caso, const char *esperado,
                          const char *obtenido) {
    int ok = (obtenido != NULL) && (strcmp(esperado, obtenido) == 0);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %-34s esperado: %-12s obtenido: %-12s %s\n",
           probadas, caso, esperado, obtenido ? obtenido : "(NULL)",
           ok ? "OK" : "FALLA");
}

static void verificar_int(const char *caso, long esperado, long obtenido) {
    int ok = (esperado == obtenido);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %-34s esperado: %-12ld obtenido: %-12ld %s\n",
           probadas, caso, esperado, obtenido, ok ? "OK" : "FALLA");
}

int main(void) {
    printf("=== AUDITORIA SESION 13 - Pila de ambitos y busqueda jerarquica ===\n");
    printf("Caso                               esperado:              obtenido:              resultado\n");

    { Ambito g; memset(&g, 0, sizeof g); g.nombre = "global"; int r = declarar(&g, "x"); verificar_int("declarar(&g, \"x\")", (long)(1), (long)(r)); }
    { Ambito g; memset(&g, 0, sizeof g); g.nombre = "global"; declarar(&g, "x"); const char *r = buscar(&g, "x"); verificar_str("buscar en el mismo ambito", "global", r); }
    { Ambito g; memset(&g, 0, sizeof g); g.nombre = "global"; declarar(&g, "x"); Ambito l; memset(&l, 0, sizeof l); l.nombre = "bloque"; l.padre = &g; const char *r = buscar(&l, "x"); verificar_str("buscar subiendo al padre", "global", r); }
    { Ambito g; memset(&g, 0, sizeof g); g.nombre = "global"; declarar(&g, "x"); Ambito l; memset(&l, 0, sizeof l); l.nombre = "bloque"; l.padre = &g; declarar(&l, "x"); const char *r = buscar(&l, "x"); verificar_str("shadowing en el hijo", "bloque", r); }
    { Ambito raiz; memset(&raiz, 0, sizeof raiz); raiz.nombre = "raiz"; Ambito medio; memset(&medio, 0, sizeof medio); medio.nombre = "medio"; medio.padre = &raiz; Ambito hoja; memset(&hoja, 0, sizeof hoja); hoja.nombre = "hoja"; hoja.padre = &medio; declarar(&raiz, "x"); const char *r = buscar(&hoja, "x"); verificar_str("tres niveles hasta la raiz", "raiz", r); }
    { Ambito g; memset(&g, 0, sizeof g); g.nombre = "global"; const char *p = buscar(&g, "z"); const char *r = p ? p : "(nulo)"; verificar_str("nombre no declarado", "(nulo)", r); }

    printf("\nRESULTADO: %d/%d pruebas superadas\n",
           probadas - fallidas, probadas);
    if (fallidas != 0) {
        printf("Aun hay %d caso(s) sin resolver. Revisa tu implementacion.\n",
               fallidas);
        return 1;
    }
    printf("TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion13.md" "\n");
    return 0;
}


In [ ]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion13_c.c artesano_sesion13_c.c -o test_artesano && ./test_artesano

In [ ]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion13_c.c ia_sesion13_c.c -o test_ia && ./test_ia

## Resultado esperado

Si tu implementación es correcta, la última línea del harness será:

```text
RESULTADO: 6/6 pruebas superadas
```

y la celda terminará sin error (código de salida 0). Compara tu salida completa
contra la de la sección 3: debe ser idéntica, línea por línea.


## Entrega

1. Copia `plantillas/auditoria_sesion13.md` a `auditorias/auditoria_sesion13.md` en tu fork.
2. Adjunta la salida del harness (las 6 líneas con `OK`).
3. Explica en el reporte crítico qué caso borde omitió la IA y cómo lo detectaste.
4. Haz commit y sincroniza tu fork.

Este laboratorio alimenta el 40% de laboratorios y es la base del Hito correspondiente
del Proyecto Integrador.
